In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

In [ ]:
from src.data_fetcher import fetch_etf_data
from src.metrics import calculate_momentum_metrics
from src.performance import calculate_core_performance

import pandas as pd
import matplotlib.pyplot as plt

# --- Universe ---

universe = [
    "SPY", "QQQ", "IWM", "DIA",

    "XLK", "SMH", "XBI", "XLE",
    "XLF", "XLY", "XLV", "XLU",
    "XLP", "XLC",

    "NVDA", "AAPL", "MSFT",
    "AMZN", "META", "AVGO",
    "TSLA", "JPM",

    "GLD", "TLT"
]



In [ ]:
# --- Fetch data ---

historical_data = fetch_etf_data(
    tickers=universe,
    start_date="2025-01-01",
    end_date="2026-05-25"
)

daily_market_returns = (
    historical_data.pct_change().dropna()
)

strategy_daily_returns = []

holding_period = 5
top_n = 3

trade_log = []



In [ ]:
# --- Weekly rebalance loop ---

for i in range(40, len(daily_market_returns) - holding_period, holding_period):

    historical_slice = historical_data.iloc[:i]

    # Regime filter:
    # only trade when SPY above 20-day moving average

    spy_price = historical_slice["SPY"].iloc[-1]
    spy_ma20 = historical_slice["SPY"].rolling(20).mean().iloc[-1]

    if spy_price < spy_ma20:
        continue

    screener = calculate_momentum_metrics(
        historical_slice,
        benchmark_ticker="SPY"
    )

    top_picks = screener.head(top_n).index.tolist()

    future_returns = daily_market_returns.iloc[
        i : i + holding_period
    ][top_picks]

    mean_daily_returns = future_returns.mean(axis=1)

    strategy_daily_returns.append(mean_daily_returns)

    trade_log.append({
        "date": historical_slice.index[-1],
        "top_picks": top_picks
    })



In [ ]:
# --- Performance series ---

strategy_perf_series = pd.concat(strategy_daily_returns)

benchmark_perf_series = daily_market_returns.loc[
    strategy_perf_series.index,
    "SPY"
]

# --- Metrics ---

core_report = calculate_core_performance(
    strategy_perf_series,
    benchmark_perf_series
)

print("\n--- STRATEGY BACKTEST RESULTS ---")

for metric, val in core_report.items():
    print(f"{metric}: {val:.2f}")

# --- Latest rankings ---

latest_screener = calculate_momentum_metrics(
    historical_data,
    benchmark_ticker="SPY"
)

print("\n--- CURRENT TOP PICKS ---")

print(
    latest_screener[
        [
            "composite_score",
            "ret_20d",
            "risk_adj_20d",
            "rel_strength_20d"
        ]
    ].head(5)
)



In [ ]:
# --- Equity curve ---

strategy_equity = (
    1 + strategy_perf_series
).cumprod()

benchmark_equity = (
    1 + benchmark_perf_series
).cumprod()

plt.figure(figsize=(12, 6))

plt.plot(strategy_equity, label="Momentum Strategy")
plt.plot(benchmark_equity, label="SPY Benchmark")

plt.title("Strategy vs SPY Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of $1")

plt.legend()

plt.show()